# Analisis Sentimen Review Produk Pertukangan Menggunakan IndoBERT
**Deskripsi Proyek:** Proyek ini bertujuan membangun model Sentiment Analysis pada ulasan produk kategori pertukangan menggunakan model IndoBERT (indobenchmark/indobert-base-p1) untuk mengklasifikasikan sentimen menjadi tiga kelas, yaitu positif, netral, dan negatif. Tahapan yang dilakukan meliputi Exploratory Data Analysis (EDA), preprocessing teks tanpa menggunakan Sastrawi, pelabelan menggunakan target asli dataset, tokenisasi, pelatihan model menggunakan class weight, evaluasi performa menggunakan Accuracy, Macro F1, Weighted F1, Error Analysis, Inference, serta penyimpanan model ke format .pkl. Model terbaik menggunakan konfigurasi learning rate 1e-5, batch size 16, epoch 6, dan warmup step 100 dengan hasil Accuracy sebesar 95,90%, Macro F1 sebesar 60,35%, dan Weighted F1 sebesar 96,04%.

# A. Import Library

In [1]:
!pip install pandas numpy scikit-learn transformers torch accelerate matplotlib -q

import pandas as pd
import numpy as np
import re
import os
import ast
import pickle
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import html

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

# B. Load Data Scraping

In [2]:
base_path = "/kaggle/input/datasets/adindaintanerlita/sentiment-pertukangan/"

df = pd.read_csv(
    base_path + "pertukangan.csv"
)

with open(
    base_path + "slangwords.csv",
    "r",
    encoding="utf-8"
) as f:

    slang_dict = ast.literal_eval(
        f.read()
    )

print(df.head())

print(
    "Jumlah data:",
    len(df)
)

print(df.info())

print(
    df.isnull().sum()
)

                                                name     category  \
0                Kuas Cat Tembok / Brush Tika 2 inch  Pertukangan   
1  [12.12 SALE][EXCLUSIVE Merabot Unik] QME- BUY ...  Pertukangan   
2  IKEA SKADIS Pengait Pegboard Bentuk L Putih Is...  Pertukangan   
3  Safety Goggles / Kacamata Safety Tali Karet AP...  Pertukangan   
4  Staple/Stapel/Straples/Staples Nail &amp; paku...  pertukangan   

                                              review  rating    target  
0  Sesuai deskripsi pelayanan cepat dan full seny...       5  Positive  
1  Wajannya sudah aku tetima tp aku kecewa bonusn...       2  Negative  
2   besi bahannya dan kuat. dan gampang pemakaiannya       5  Positive  
3                            Barang sesuai deskripsi       5  Positive  
4                    gggggggggoooooooodddddddddddddd       5  Positive  
Jumlah data: 10000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column    Non-Nu

# C. Exploratory Data Analysis (EDA)

In [3]:
print("Ukuran dataset:", df.shape)

print("\nNama kolom:")
print(df.columns)

print("\nMissing value:")
print(df.isnull().sum())

print("\nDuplikasi:")
print(df.duplicated().sum())

print("\nDistribusi rating:")
print(
    df["rating"]
    .value_counts()
    .sort_index()
)

print("\nDistribusi target:")
print(
    df["target"]
    .value_counts()
)

Ukuran dataset: (10000, 5)

Nama kolom:
Index(['name', 'category', 'review', 'rating', 'target'], dtype='object')

Missing value:
name        0
category    0
review      0
rating      0
target      0
dtype: int64

Duplikasi:
0

Distribusi rating:
rating
1     108
2      49
3     223
4     764
5    8856
Name: count, dtype: int64

Distribusi target:
target
Positive    9620
Neutral      223
Negative     157
Name: count, dtype: int64


# D. Preprocessing

In [4]:
emoji_dict = {
    "😊": "senang", "😁": "senang", "😍": "suka",
    "🥰": "suka", "😭": "kecewa", "😢": "sedih",
    "😡": "marah", "😤": "kesal", "🤩": "senang",
    "🔥": "bagus", "👍": "bagus", "👎": "jelek",
    "💔": "kecewa", "❤️": "suka", "❤": "suka"
}

def clean_text(text):

    text = str(text).lower()

    text = html.unescape(text)

    text = re.sub(r'\\n|\\t|\\r|\\b|\\f|\\v', ' ', text)
    text = re.sub(r'\\u[0-9a-fA-F]{4}', ' ', text)
    text = re.sub(r'\\x[0-9a-fA-F]{2}', ' ', text)

    text = re.sub(r'http\S+|www\S+', '', text)

    for emo, meaning in emoji_dict.items():
        text = text.replace(emo, " " + meaning + " ")

    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    words = text.split()
    words = [slang_dict.get(w, w) for w in words]

    text = " ".join(words)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

df = df.dropna(subset=["review"])

df["clean_text"] = df["review"].apply(clean_text)

df = df[df["clean_text"] != ""]

print(df[["review", "clean_text", "rating"]].head())

print("\nKolom dataset:")
print(df.columns)

if "target" in df.columns:
    print("\nDistribusi Target:")
    print(df["target"].value_counts())

                                              review  \
0  Sesuai deskripsi pelayanan cepat dan full seny...   
1  Wajannya sudah aku tetima tp aku kecewa bonusn...   
2   besi bahannya dan kuat. dan gampang pemakaiannya   
3                            Barang sesuai deskripsi   
4                    gggggggggoooooooodddddddddddddd   

                                          clean_text  rating  
0  sesuai deskripsi pelayanan cepat dan penuh senyum       5  
1  wajannya sudah aku tetima tapi aku kecewa bonu...       2  
2    besi bahannya dan kuat dan gampang pemakaiannya       5  
3                            barang sesuai deskripsi       5  
4                    gggggggggoooooooodddddddddddddd       5  

Kolom dataset:
Index(['name', 'category', 'review', 'rating', 'target', 'clean_text'], dtype='object')

Distribusi Target:
target
Positive    9610
Neutral      223
Negative     157
Name: count, dtype: int64


# E. Labeling

In [5]:
df["sentiment"] = df["target"].str.lower()

df["sentiment"] = df["sentiment"].replace({
    "positive": "positif",
    "neutral": "netral",
    "negative": "negatif"
})

print("Distribusi Sentimen:")
print(df["sentiment"].value_counts())

Distribusi Sentimen:
sentiment
positif    9610
netral      223
negatif     157
Name: count, dtype: int64


# E1. Label Encoding

In [6]:
sentiment_labels = sorted(df["sentiment"].unique())

label_map = {
    label: i
    for i, label in enumerate(sentiment_labels)
}

sentiment_map = {
    v: k
    for k, v in label_map.items()
}

df["label"] = df["sentiment"].map(label_map)

print("Label Map:")
print(label_map)

Label Map:
{'negatif': 0, 'netral': 1, 'positif': 2}


# F. Split Data

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Distribusi Train:")
print(y_train.value_counts())

print("\nDistribusi Test:")
print(y_test.value_counts())

Distribusi Train:
label
2    7688
1     178
0     126
Name: count, dtype: int64

Distribusi Test:
label
2    1922
1      45
0      31
Name: count, dtype: int64


# G. Tokenizer dan Dataset

In [8]:
model_name = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

lengths = df["clean_text"].str.split().apply(len)

max_len = int(np.percentile(lengths, 95))
max_len = min(max_len, 128)

print("Max Length:", max_len)

train_enc = tokenizer(
    list(X_train),
    truncation=True,
    padding=True,
    max_length=max_len
)

test_enc = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=max_len
)

class SentimentDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {
            k: torch.tensor(v[idx])
            for k, v in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(train_enc, y_train)
test_dataset = SentimentDataset(test_enc, y_test)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Max Length: 27


# H. Class Weight

In [9]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

print("Class Weights:", class_weights)

Class Weights: tensor([21.1429, 14.9663,  0.3465])


# I. Mertics

In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

# J. Custom Trainer dengan Class Weight

In [11]:
class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]

        weights = class_weights.to(logits.device)

        loss_fct = nn.CrossEntropyLoss(weight=weights)

        num_labels = logits.shape[-1]

        loss = loss_fct(
            logits.view(-1, num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

# K. Training

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(sentiment_labels),
    id2label={
        idx: label
        for idx, label in enumerate(sentiment_labels)
    },
    label2id={
        label: idx
        for idx, label in enumerate(sentiment_labels)
    },
    ignore_mismatched_sizes=True
)

training_args = TrainingArguments(
    output_dir="./results_pertukangan_best_v2",

    learning_rate=1e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=6,

    eval_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=1,

    load_best_model_at_end=True,

    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_steps=100,

    warmup_steps=100,

    weight_decay=0.01,

    fp16=torch.cuda.is_available(),

    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

trainer.train()

eval_result = trainer.evaluate()

print("\nHASIL EVALUASI:")
print(eval_result)

best_model = trainer.model

best_result = {
    "learning_rate": 1e-5,
    "batch_size": 16,
    "epoch": 6,
    "warmup_steps": 100,
    "accuracy": eval_result["eval_accuracy"],
    "f1_weighted": eval_result["eval_f1_weighted"],
    "f1_macro": eval_result["eval_f1_macro"]
}

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,F1 Macro
1,0.865275,0.843329,0.959459,0.960948,0.611920
2,0.656183,1.136092,0.964965,0.961029,0.552397
3,0.612288,0.960022,0.951952,0.955279,0.565184


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


HASIL EVALUASI:
{'eval_loss': 0.8441649675369263, 'eval_accuracy': 0.958958958958959, 'eval_f1_weighted': 0.9604334959870954, 'eval_f1_macro': 0.6034790392184758, 'eval_runtime': 4.5236, 'eval_samples_per_second': 441.68, 'eval_steps_per_second': 13.927, 'epoch': 3.0}


# L. Evaluasi Model Terbaik

In [13]:
best_model = trainer.model
best_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model.to(device)

all_preds = []
all_labels = []

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

with torch.no_grad():
    for batch in test_loader:
        labels = batch["labels"].cpu().numpy()

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        outputs = best_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        preds = torch.argmax(
            outputs.logits,
            dim=1
        ).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels)

y_pred = np.array(all_preds)
y_true = np.array(all_labels)

print("\nCLASSIFICATION REPORT:")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(sentiment_labels))),
    target_names=sentiment_labels
))

print("\nCONFUSION MATRIX:")
print(confusion_matrix(y_true, y_pred))

print("\nF1-SCORE PER CLASS:")
f1_per_class = f1_score(
    y_true,
    y_pred,
    average=None
)

for label, score in zip(sentiment_labels, f1_per_class):
    status = "AMAN" if score >= 0.75 else "PERLU DITINGKATKAN"
    print(f"{label:10s}: F1 = {score:.4f} | {status}")

print(f"\nAccuracy   : {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1   : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted'):.4f}")


CLASSIFICATION REPORT:
              precision    recall  f1-score   support

     negatif       0.49      0.55      0.52        31
      netral       0.29      0.33      0.31        45
     positif       0.99      0.98      0.98      1922

    accuracy                           0.96      1998
   macro avg       0.59      0.62      0.60      1998
weighted avg       0.96      0.96      0.96      1998


CONFUSION MATRIX:
[[  17    6    8]
 [  10   15   20]
 [   8   30 1884]]

F1-SCORE PER CLASS:
negatif   : F1 = 0.5152 | PERLU DITINGKATKAN
netral    : F1 = 0.3125 | PERLU DITINGKATKAN
positif   : F1 = 0.9828 | AMAN

Accuracy   : 0.9590
Macro F1   : 0.6035
Weighted F1: 0.9604


# M.  Error Analysis

In [14]:
best_model = trainer.model
best_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model.to(device)

all_preds = []
all_labels = []

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

with torch.no_grad():
    for batch in test_loader:
        labels = batch["labels"].cpu().numpy()

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        outputs = best_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        preds = torch.argmax(
            outputs.logits,
            dim=1
        ).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels)

y_pred = np.array(all_preds)
y_true = np.array(all_labels)

print("\nCLASSIFICATION REPORT:")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(sentiment_labels))),
    target_names=sentiment_labels
))

print("\nCONFUSION MATRIX:")
print(confusion_matrix(y_true, y_pred))

print("\nF1-SCORE PER CLASS:")
f1_per_class = f1_score(
    y_true,
    y_pred,
    average=None
)

for label, score in zip(sentiment_labels, f1_per_class):
    status = "AMAN" if score >= 0.75 else "PERLU DITINGKATKAN"
    print(f"{label:10s}: F1 = {score:.4f} | {status}")

print(f"\nAccuracy   : {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1   : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted'):.4f}")


CLASSIFICATION REPORT:
              precision    recall  f1-score   support

     negatif       0.49      0.55      0.52        31
      netral       0.29      0.33      0.31        45
     positif       0.99      0.98      0.98      1922

    accuracy                           0.96      1998
   macro avg       0.59      0.62      0.60      1998
weighted avg       0.96      0.96      0.96      1998


CONFUSION MATRIX:
[[  17    6    8]
 [  10   15   20]
 [   8   30 1884]]

F1-SCORE PER CLASS:
negatif   : F1 = 0.5152 | PERLU DITINGKATKAN
netral    : F1 = 0.3125 | PERLU DITINGKATKAN
positif   : F1 = 0.9828 | AMAN

Accuracy   : 0.9590
Macro F1   : 0.6035
Weighted F1: 0.9604


# N. Inferene

In [15]:
def predict_sentiment(text, show_confidence=False):

    text_clean = clean_text(text)

    encoding = tokenizer(
        [text_clean],
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_len
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    best_model.to(device)
    best_model.eval()

    encoding = {
        k: v.to(device)
        for k, v in encoding.items()
    }

    with torch.no_grad():
        output = best_model(**encoding)

    pred = torch.argmax(
        output.logits,
        dim=1
    ).item()

    if show_confidence:
        probs = torch.softmax(
            output.logits,
            dim=1
        ).squeeze().tolist()

        print("\nConfidence per kelas:")

        for label, prob in zip(sentiment_labels, probs):
            print(f"{label:10s}: {prob:.2%}")

    return sentiment_map[pred]


test_samples = [
    "Bor listriknya bagus banget, kuat, dan baterainya tahan lama",
    "Obeng set ini biasa saja, kualitasnya standar tapi masih bisa dipakai",
    "Gerindanya rusak, cepat panas, dan tidak sesuai deskripsi"
]

print("\nHASIL INFERENCE:")

for sample in test_samples:
    hasil = predict_sentiment(
        sample,
        show_confidence=True
    )

    print("\nTeks     :", sample)
    print("Sentimen :", hasil)
    print("-" * 50)


HASIL INFERENCE:

Confidence per kelas:
negatif   : 0.19%
netral    : 1.49%
positif   : 98.31%

Teks     : Bor listriknya bagus banget, kuat, dan baterainya tahan lama
Sentimen : positif
--------------------------------------------------

Confidence per kelas:
negatif   : 1.30%
netral    : 8.46%
positif   : 90.23%

Teks     : Obeng set ini biasa saja, kualitasnya standar tapi masih bisa dipakai
Sentimen : positif
--------------------------------------------------

Confidence per kelas:
negatif   : 49.35%
netral    : 44.01%
positif   : 6.64%

Teks     : Gerindanya rusak, cepat panas, dan tidak sesuai deskripsi
Sentimen : negatif
--------------------------------------------------


# O. Simpan Model ke .pkl

In [16]:
SAVE_DIR = "/kaggle/working/model_sentiment_pertukangan"

os.makedirs(SAVE_DIR, exist_ok=True)

best_model_cpu = best_model.to("cpu")

best_model_cpu.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

best_result = {
    "learning_rate": 1e-5,
    "batch_size": 16,
    "epoch": 6,
    "warmup_steps": 100,
    "accuracy": accuracy_score(y_true, y_pred),
    "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
    "f1_macro": f1_score(y_true, y_pred, average="macro")
}

model_bundle = {
    "model_state": best_model_cpu.state_dict(),
    "config": best_model_cpu.config,
    "tokenizer": tokenizer,
    "label_map": label_map,
    "sentiment_map": sentiment_map,
    "max_len": max_len,
    "best_experiment": best_result
}

file_path = "/kaggle/working/model_sentiment_pertukangan_indobert.pkl"

with open(file_path, "wb") as f:
    pickle.dump(model_bundle, f)

df_save = df[
    [
        "review",
        "clean_text",
        "rating",
        "sentiment",
        "label"
    ]
]

df_save.to_csv(
    "/kaggle/working/clean_dataset_sentiment_pertukangan.csv",
    index=False
)

print("\nMODEL SAVED")
print("File model:", file_path)
print("File dataset bersih: /kaggle/working/clean_dataset_sentiment_pertukangan.csv")

print("\nDaftar file di /kaggle/working:")
print(os.listdir("/kaggle/working"))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


MODEL SAVED
File model: /kaggle/working/model_sentiment_pertukangan_indobert.pkl
File dataset bersih: /kaggle/working/clean_dataset_sentiment_pertukangan.csv

Daftar file di /kaggle/working:
['.virtual_documents', 'model_sentiment_pertukangan_indobert.pkl', 'model_sentiment_pertukangan', 'results_pertukangan_best_v2', 'clean_dataset_sentiment_pertukangan.csv']
